# Semana 21 — Avaliando o Fine-Tuned Model
**Fase 4: Parameter-Efficient Fine-Tuning (PEFT)** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/notebooks/semana_21_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-21-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/docs/Semana-21-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_21_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install peft

In [ ]:
import torch, json, time, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from sklearn.metrics import classification_report
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
ALVOS = ["q_proj", "k_proj", "v_proj", "o_proj"]

## Passo 1 — Reproduzir o fine-tuned model da Semana 19
Dataset e utilitários já prontos; o treino leva poucos minutos na T4.

In [ ]:
# --- Dados brutos: chamados de TI fictícios, com a sujeira típica de dados reais ---
import random
SISTEMA_TICKETS = ('Você é um classificador de chamados de TI. Responda apenas com um JSON com as chaves '
                   '"categoria", "prioridade" e "resumo".')
CATEGORIAS = ["acesso", "hardware", "software", "rede", "financeiro"]; PRIORIDADES = ["baixa", "media", "alta"]
def gerar_tickets_brutos(n=600, seed=0):
    rnd = random.Random(seed)
    modelos = {
        "acesso": [("Não consigo entrar no {x}, aparece senha inválida.", "Falha de login no {x}"),
                   ("Meu usuário foi bloqueado no {x} depois de várias tentativas.", "Usuário bloqueado no {x}")],
        "hardware": [("O {x} não liga desde ontem.", "{x} não liga"), ("A tela do {x} está piscando sem parar.", "Tela do {x} com defeito")],
        "software": [("O {x} fecha sozinho quando abro um arquivo grande.", "{x} fecha inesperadamente"),
                     ("Preciso instalar o {x} na minha máquina.", "Instalação do {x}")],
        "rede": [("A internet do {x} está muito lenta.", "Lentidão de rede no {x}"), ("O wifi do {x} cai a cada dez minutos.", "Queda de wifi no {x}")],
        "financeiro": [("Minha nota de reembolso de {x} reais não foi paga.", "Reembolso de {x} reais pendente"),
                       ("Fui cobrado em duplicidade em {x} reais na fatura.", "Cobrança duplicada de {x} reais")],
    }
    slots = {"acesso": ["SAP", "portal RH", "CRM", "e-mail corporativo", "VPN"], "hardware": ["notebook", "monitor", "teclado", "headset", "dock"],
             "software": ["Excel", "Power BI", "Teams", "VS Code", "Outlook"], "rede": ["3º andar", "escritório de Lisboa", "auditório", "sala de reuniões", "home office"],
             "financeiro": ["150", "480", "1200", "75", "320"]}
    urgencia = {"alta": ["Isso está parando toda a equipe!", "É urgente, tenho uma entrega hoje."],
                "media": ["Preciso disso resolvido esta semana.", "Está atrapalhando meu trabalho."],
                "baixa": ["Sem pressa.", "Quando puderem, por favor."]}
    saudacoes = ["Olá,", "Bom dia!", "Oi pessoal,", "Prezados,", ""]
    nomes = ["ana.souza", "joao.lima", "carla.mendes", "pedro.alves", "rita.costa"]
    linhas = []
    for _ in range(n):
        cat = rnd.choice(CATEGORIAS); texto, resumo = rnd.choice(modelos[cat]); x = rnd.choice(slots[cat]); prio = rnd.choice(PRIORIDADES)
        corpo = f"{rnd.choice(saudacoes)} {texto.format(x=x)} {rnd.choice(urgencia[prio])}"
        if rnd.random() < 0.5: corpo += f"  Att, {rnd.choice(nomes)}@empresa.com, tel (11) 9{rnd.randint(1000, 9999)}-{rnd.randint(1000, 9999)}"
        if rnd.random() < 0.2: corpo = "   " + corpo.replace(" ", "  ", 2)
        rot_cat = rnd.choice([cat, cat, cat.upper(), f" {cat} "]); rot_prio = rnd.choice([prio, prio, prio.capitalize(), "média" if prio == "media" else prio])
        linhas.append({"texto": corpo, "categoria": rot_cat, "prioridade": rot_prio, "resumo": resumo.format(x=x)})
    linhas += [dict(rnd.choice(linhas)) for _ in range(n // 12)]                      # duplicatas exatas
    linhas += [{"texto": "", "categoria": "acesso", "prioridade": "baixa", "resumo": ""} for _ in range(n // 40)]   # vazios
    rnd.shuffle(linhas); return linhas

# --- Pipeline de preparação (construído passo a passo na Semana 17) ---
import re, json, unicodedata
def preparar_tickets(n=600, seed=0):
    """Devolve (treino, val, teste): listas de {"messages": [system, user, assistant]}."""
    norm = lambda s: unicodedata.normalize("NFKD", s.strip().lower()).encode("ascii", "ignore").decode()
    vistos, exemplos = set(), []
    for l in gerar_tickets_brutos(n, seed):
        t = re.sub(r"\s+", " ", l["texto"]).strip()
        t = re.sub(r"[\w.]+@[\w.]+", "<EMAIL>", t); t = re.sub(r"\(\d{2}\)\s*9?\d{4}-\d{4}", "<TELEFONE>", t)
        if not t or t in vistos: continue
        vistos.add(t)
        alvo = {"categoria": norm(l["categoria"]), "prioridade": norm(l["prioridade"]), "resumo": l["resumo"]}
        exemplos.append({"messages": [{"role": "system", "content": SISTEMA_TICKETS}, {"role": "user", "content": t},
                                      {"role": "assistant", "content": json.dumps(alvo, ensure_ascii=False)}]})
    random.Random(seed).shuffle(exemplos); a, b = int(0.8 * len(exemplos)), int(0.9 * len(exemplos))
    return exemplos[:a], exemplos[a:b], exemplos[b:]

treino, val, teste = preparar_tickets()
# --- Utilitários de fine-tuning (escritos passo a passo na Semana 19) ---
import json, random, torch
def gerar(modelo, tok, mensagens, max_new_tokens=64):
    prompt = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    ent = tok(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    with torch.no_grad():
        out = modelo.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=pad)
    return tok.decode(out[0, ent["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def codificar(tok, mensagens, max_len=256):
    """ids = prompt + resposta; labels = -100 no prompt (só a resposta entra na perda)."""
    prompt = tok.apply_chat_template(mensagens[:-1], tokenize=False, add_generation_prompt=True)
    ids_p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids_r = tok(mensagens[-1]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return (ids_p + ids_r)[:max_len], ([-100] * len(ids_p) + ids_r)[:max_len]

def colar(tok, exemplos, device):
    cods = [codificar(tok, e["messages"]) for e in exemplos]; T = max(len(i) for i, _ in cods)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.tensor([i + [pad] * (T - len(i)) for i, _ in cods]); lab = torch.tensor([l + [-100] * (T - len(l)) for _, l in cods])
    mask = torch.tensor([[1] * len(i) + [0] * (T - len(i)) for i, _ in cods])
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device), "labels": lab.to(device)}

def treinar(modelo, tok, dados, epocas=2, lr=2e-4, lote=8, log=20):
    device = modelo.device; amp = device.type == "cuda"
    opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    scaler = torch.amp.GradScaler(enabled=amp); hist = []; modelo.train(); rnd = random.Random(0)
    for ep in range(epocas):
        ordem = list(range(len(dados))); rnd.shuffle(ordem)
        for n, i in enumerate(range(0, len(ordem), lote)):
            b = colar(tok, [dados[j] for j in ordem[i:i + lote]], device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                loss = modelo(**b).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); hist.append(loss.item())
            if n % log == 0: print(f"época {ep} passo {n:3d} | loss {loss.item():.3f}")
    modelo.eval(); return hist

@torch.no_grad()
def perda(modelo, tok, dados, lote=8):
    modelo.eval(); device = modelo.device; tot = 0.0
    for i in range(0, len(dados), lote):
        b = colar(tok, dados[i:i + lote], device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            tot += modelo(**b).loss.item() * len(dados[i:i + lote])
    return tot / len(dados)

def avaliar_extracao(modelo, tok, dados, n=40):
    """Gera a resposta para cada exemplo e compara com o gabarito. Devolve lista de dicts."""
    linhas = []
    for e in dados[:n]:
        ouro = json.loads(e["messages"][-1]["content"]); bruto = gerar(modelo, tok, e["messages"][:-1])
        try:
            p = json.loads(bruto[bruto.index("{"): bruto.rindex("}") + 1]); ok = isinstance(p, dict)
        except Exception:
            p, ok = {}, False
        linhas.append({"json_valido": ok, "categoria_ok": p.get("categoria") == ouro["categoria"], "prioridade_ok": p.get("prioridade") == ouro["prioridade"],
                       "tudo_ok": ok and all(p.get(k) == ouro[k] for k in ouro), "cat_ouro": ouro["categoria"], "cat_pred": p.get("categoria"), "bruto": bruto})
    return linhas
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = get_peft_model(AutoModelForCausalLM.from_pretrained(MODELO).to(device),
                        LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=ALVOS, task_type="CAUSAL_LM"))
hist = treinar(modelo, tok, treino, epocas=2)

## Passo 2 — Três concorrentes
- **Zero-shot:** base model (adapter desligado com `modelo.disable_adapter()`).
- **Few-shot:** base model com 3 exemplos de treino inseridos no prompt.
- **Fine-tuned:** adapter ligado.

Escreva `com_exemplos`: recebe um exemplo e devolve o mesmo exemplo com as mensagens `system, (user, assistant) × 3, user, assistant`.

In [ ]:
EXEMPLOS = treino[:3]

def com_exemplos(e):
    # [SEU CÓDIGO AQUI] monte a lista de mensagens com os 3 exemplos antes da pergunta

assert [m["role"] for m in com_exemplos(val[0])["messages"]] == ["system"] + ["user", "assistant"] * 3 + ["user", "assistant"]

In [ ]:
N = 40
with modelo.disable_adapter():
    r_zero = pd.DataFrame(avaliar_extracao(modelo, tok, teste, N))
    r_few = pd.DataFrame(avaliar_extracao(modelo, tok, [com_exemplos(e) for e in teste], N))
r_ft = pd.DataFrame(avaliar_extracao(modelo, tok, teste, N))

## Passo 3 — Tabela comparativa
Uma linha por concorrente com a média de `json_valido`, `categoria_ok`, `prioridade_ok` e `tudo_ok`.

In [ ]:
cols = ["json_valido", "categoria_ok", "prioridade_ok", "tudo_ok"]
# [SEU CÓDIGO AQUI] monte o DataFrame `comparativo`
comparativo.round(3)

## Passo 4 — Relatório por classe
Use `classification_report` para a categoria. Previsões ausentes ou inválidas viram a classe `"invalido"`.

In [ ]:
def relatorio(r):
    # [SEU CÓDIGO AQUI] substitua previsões fora de CATEGORIAS por "invalido" e imprima o classification_report

print("=== few-shot ==="); relatorio(r_few); print("=== ajustado ==="); relatorio(r_ft)

## Passo 5 — Custo: tokens e latência
Conte os tokens de entrada de cada abordagem e meça a latência média por requisição.

In [ ]:
def tokens_entrada(e):
    # [SEU CÓDIGO AQUI] número de tokens do prompt formatado

def latencia(dados, n=8):
    t0 = time.perf_counter()
    for e in dados[:n]: gerar(modelo, tok, e["messages"][:-1])
    return (time.perf_counter() - t0) / n * 1000

few = [com_exemplos(e) for e in teste]
custo = pd.DataFrame({"tokens de entrada": {"zero-shot / ajustado": tokens_entrada(teste[0]), "few-shot (3)": tokens_entrada(few[0])},
                      "latência (ms)": {"zero-shot / ajustado": latencia(teste), "few-shot (3)": latencia(few)}}).round(0); custo

Com 1 milhão de requisições por mês, quantos tokens de entrada a mais o few-shot consome?

## Passo 6 — Loss e perplexity em held-out data

In [ ]:
with modelo.disable_adapter(): p_base = perda(modelo, tok, teste)
p_ft = perda(modelo, tok, teste)
print(f"base    : perda {p_base:.3f} | perplexidade {torch.exp(torch.tensor(p_base)).item():.2f}")
print(f"ajustado: perda {p_ft:.3f} | perplexidade {torch.exp(torch.tensor(p_ft)).item():.2f}")

## Passo 7 — Out-of-distribution
Chamados escritos de outro jeito, sem os modelos de frase do treino. Monte os exemplos no formato de chat e avalie.

In [ ]:
novos = [("meu acesso ao sistema de vendas foi negado, tenho cliente esperando agora", "acesso", "alta"),
         ("a impressora do financeiro está engasgando papel, nada grave", "hardware", "baixa"),
         ("o aplicativo de planilhas trava toda vez que salvo, isso me atrasa bastante", "software", "media"),
         ("conexão caindo na filial do Porto, ninguém consegue trabalhar!!", "rede", "alta"),
         ("ainda não recebi o valor da viagem de março, podem verificar quando der?", "financeiro", "baixa"),
         ("esqueci minha credencial do portal e preciso entrar até sexta", "acesso", "media"),
         ("mouse sem fio parou de responder", "hardware", "baixa"),
         ("a fatura deste mês veio com um valor cobrado duas vezes, preciso resolver esta semana", "financeiro", "media")]
# [SEU CÓDIGO AQUI] construa `ood` no mesmo formato de `teste` (resumo vazio no gabarito)
r_ood = pd.DataFrame(avaliar_extracao(modelo, tok, ood, len(ood)))
print(pd.DataFrame({"teste (mesma distribuição)": r_ft[cols[:3]].mean(), "fora da distribuição": r_ood[cols[:3]].mean()}).round(3))
r_ood[["cat_ouro", "cat_pred", "bruto"]]

Qual campo generalizou melhor: categoria ou prioridade? O que isso diz sobre o que o modelo aprendeu?

## Passo 8 — Forgetting
A mesma pergunta geral, com o adapter desligado e ligado.

In [ ]:
geral = [{"role": "user", "content": "Explique em uma frase o que é uma rede neural."}]
with modelo.disable_adapter(): print("BASE    :", gerar(modelo, tok, geral, 60))
print("AJUSTADO:", gerar(modelo, tok, geral, 60))

## Passo 9 — Merge para deployment
Faça o merge do adapter, confirme que a saída é a mesma e salve o modelo completo.

In [ ]:
antes = gerar(modelo, tok, teste[0]["messages"][:-1])
# [SEU CÓDIGO AQUI] faça o merge do adapter no base model
assert gerar(fundido, tok, teste[0]["messages"][:-1]) == antes
print("tipo após a fusão:", type(fundido).__name__, "| módulos LoRA restantes:", sum("lora" in n for n, _ in fundido.named_modules()))
fundido.save_pretrained("modelo-chamados-fundido"); tok.save_pretrained("modelo-chamados-fundido")

## Fechamento
Escreva o `MODEL_CARD.md` pedido no entregável. Inclua a decisão: para este caso de uso, você recomendaria zero-shot, few-shot ou fine-tuning? Com base em quais números?